In [3]:
import numpy as np
import os
import matplotlib.pyplot as plt

# 1. 权威数据初始化 (如果文件不存在则创建)
TRUE_80_LIST = [
    14.13472514, 21.02203964, 25.01085758, 30.42487613, 32.93506159, 37.58617816, 40.91871901, 43.32707328, 48.00515088, 49.77383248,
    52.97032148, 56.44624770, 59.34704400, 60.83177852, 65.11254405, 67.07981053, 69.54640171, 72.06715767, 75.70469070, 77.14484007,
    79.33737502, 82.91038085, 84.73549298, 87.42527461, 88.80911121, 92.49189927, 94.65134404, 95.87063423, 98.83119422, 101.31785101,
    103.72553804, 105.44662305, 107.16861118, 111.02953554, 111.87465918, 114.32022091, 116.22668032, 118.79072351, 121.37012500, 122.94682930,
    124.25681855, 127.51668388, 129.57870420, 131.08768886, 133.49773720, 134.75650975, 138.11604205, 139.73620900, 141.12370740, 143.11184581,
    146.00098249, 147.42276535, 150.05352042, 150.92525761, 153.56503606, 156.11290929, 157.59759182, 158.84998817, 161.18896414, 163.03070969,
    165.53706919, 167.18139516, 169.09451542, 171.21633519, 172.26833441, 175.74415446, 177.34856331, 178.67750529, 182.20707848, 184.87446785,
    185.59878368, 187.22892258, 189.41615867, 192.02665636, 193.07692257, 195.26531310, 196.57648210, 199.15711235, 201.26475143, 202.49359451
]
TRUE_GAMMAS_80 = np.array(TRUE_80_LIST)

def harvest_80_modes():
    files = [f for f in os.listdir('.') if f.startswith('res_k_') and f.endswith('_steps10t10.npy')]
    
    all_k = []
    all_phases = []
    
    print(f"开始检查 {len(files)} 个文件...")
    for f in sorted(files):
        try:
            k_val = float(f.split('_')[2])
            phases = np.load(f)
            # 鲁棒性检查：只有点数 >= 80 的文件才参与收割
            if len(phases) >= 80:
                all_k.append(k_val)
                all_phases.append(phases[:80])
            else:
                print(f"Skipping {f}: Only has {len(phases)} points.")
        except Exception as e:
            print(f"Error loading {f}: {e}")
    
    if not all_phases:
        print("错误：没有一个文件包含 80 个以上的点！请检查数据。")
        return

    all_k = np.array(all_k)
    all_phases = np.array(all_phases) # 此时 shape 为 (N, 80)，不再 inhomogeneous [cite: 2026-02-05]
    
    best_ks = []
    min_errors = []
    for i in range(80):
        # 归一化对齐：利用模式 1 校准
        scalings = TRUE_GAMMAS_80[0] / all_phases[:, 0]
        current_mode_sim = all_phases[:, i] * scalings
        
        # 寻找误差绝对值最小的 k
        errors = current_mode_sim - TRUE_GAMMAS_80[i]
        best_idx = np.argmin(np.abs(errors))
        
        best_ks.append(all_k[best_idx])
        min_errors.append(errors[best_idx])
    
    # 绘图逻辑保持不变...
    plt.figure(figsize=(15, 6))
    plt.scatter(range(1, 81), best_ks, color='black', s=25, label='Optimal k per Mode')
    plt.axhline(y=4.7, color='r', linestyle='--', label='UV Limit (4.7000)')
    
    # 趋势线
    z = np.polyfit(range(1, 81), best_ks, 3)
    p = np.poly1d(z)
    plt.plot(range(1, 81), p(range(1, 81)), "b-", alpha=0.6, label='RG Flow Trend')
    
    plt.title('The 80-Mode Renormalization Flow Analysis', fontsize=16)
    plt.xlabel('Riemann Zero Index (n)')
    plt.ylabel('Optimal Coupling Constant k')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('k_flow_80_modes_final.png', dpi=300)
    plt.show()
    
    print(f">>> 成功收割 {len(all_k)} 个有效文件！趋势图已保存。")

if __name__ == "__main__":
    harvest_80_modes()

开始检查 64 个文件...
Skipping res_k_10.0533_steps10t10.npy: Only has 26 points.
Skipping res_k_10.1808_steps10t10.npy: Only has 25 points.
Skipping res_k_10.3083_steps10t10.npy: Only has 26 points.
Skipping res_k_10.4357_steps10t10.npy: Only has 27 points.
Skipping res_k_10.5632_steps10t10.npy: Only has 29 points.
Skipping res_k_10.6906_steps10t10.npy: Only has 27 points.
Skipping res_k_10.8181_steps10t10.npy: Only has 32 points.
Skipping res_k_10.9456_steps10t10.npy: Only has 28 points.
Skipping res_k_11.0730_steps10t10.npy: Only has 35 points.
Skipping res_k_11.2005_steps10t10.npy: Only has 33 points.
Skipping res_k_11.3279_steps10t10.npy: Only has 29 points.
Skipping res_k_11.4554_steps10t10.npy: Only has 26 points.
Skipping res_k_11.5829_steps10t10.npy: Only has 24 points.
Skipping res_k_11.7103_steps10t10.npy: Only has 26 points.
Skipping res_k_11.8378_steps10t10.npy: Only has 24 points.
Skipping res_k_11.9652_steps10t10.npy: Only has 26 points.
Skipping res_k_12.0927_steps10t10.npy: On